# MSS Output Explorer

Interaktive Visualisierung der Simulations-Outputs mit **Plotly**.

- Nutzt standardmässig den **neusten regulären** Output-Lauf (`outputs/YYYYMMDD_HHMMSS`, ohne Suffix wie `_Sweep` / `_Single`).
- Manuell überschreibbar via `OUTPUT_DIR_NAME`.
- Heatmaps und weitere Plots haben einen **Tages-Schieberegler** (+ Play/Pause), um die Entwicklung über die Zeit statt nur den letzten Tag zu sehen.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Konfiguration ===
# None  -> automatisch der neuste reguläre Output (outputs/YYYYMMDD_HHMMSS ohne Suffix).
# str   -> manueller Ordnername, z.B. "20260604_224617".
OUTPUT_DIR_NAME = None
DAY_STEP = 1          # Schrittweite des Tages-Sliders (1 = jeder Tag; höher = leichter/schneller)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

_PLAIN = re.compile(r"^\d{8}_\d{6}$")  # nur Zeitstempel, nichts dahinter

def resolve_output_dir(name=None):
    if name:
        p = OUTPUTS_DIR / name
        if not p.exists():
            raise FileNotFoundError(f"Output-Ordner nicht gefunden: {p}")
        return p
    plain = sorted(d for d in OUTPUTS_DIR.iterdir() if d.is_dir() and _PLAIN.match(d.name))
    if not plain:
        raise FileNotFoundError("Kein regulärer Output (YYYYMMDD_HHMMSS) in outputs/ gefunden.")
    return plain[-1]

OUTPUT_DIR = resolve_output_dir(OUTPUT_DIR_NAME)
DATA_DIR = OUTPUT_DIR / "data"
print("Verwendeter Output:", OUTPUT_DIR.name)

In [ ]:
def load(name):
    p = DATA_DIR / f"{name}.parquet"
    if not p.exists():
        print(f"  (fehlt: {name}.parquet)")
        return None
    return pd.read_parquet(p)

macro_daily       = load("macro_daily")
macro_by_hospital = load("macro_daily_by_hospital")
cell_daily        = load("macro_cell_daily")
micro_daily       = load("micro_daily")

print("Geladene Tabellen:")
for nm, df in [("macro_daily", macro_daily), ("macro_daily_by_hospital", macro_by_hospital),
               ("macro_cell_daily", cell_daily), ("micro_daily", micro_daily)]:
    if df is not None:
        print(f"  {nm:24s} {len(df):>7d} Zeilen, Tage {int(df['day'].min())}-{int(df['day'].max())}")

In [ ]:
def _slider_and_buttons(days):
    """Slider unten + Play/Pause links daneben (überdeckt keine (Subplot-)Titel)."""
    steps = [dict(method="animate", label=str(d),
                  args=[[str(d)], dict(mode="immediate",
                                       frame=dict(duration=0, redraw=True),
                                       transition=dict(duration=0))])
             for d in days]
    play = dict(label="▶", method="animate",
                args=[None, dict(frame=dict(duration=120, redraw=True),
                                 fromcurrent=True, transition=dict(duration=0))])
    pause = dict(label="⏸", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
    sliders = [dict(active=0, currentvalue=dict(prefix="Tag: "), pad=dict(t=50),
                    x=0.12, len=0.88, steps=steps)]
    updatemenus = [dict(type="buttons", direction="left", showactive=False,
                        x=0.10, xanchor="right", y=0, yanchor="top", pad=dict(t=50, r=10),
                        buttons=[play, pause])]
    return sliders, updatemenus

## Gefacettete Per-Spital-Heatmap mit Tages-Schieberegler

`hospital_grid_heatmap_with_day_slider(...)` zeigt — im Layout des `department_grid`-Plots — **ein Gitter pro Spital** (3×2), gemeinsame Farbskala, ICU-Reihen oben. Der Slider verstellt den Tag und aktualisiert alle Spitäler gleichzeitig. Hover zeigt Wert und Patientenzahl `n` je Zelle.

In [ ]:
def hospital_grid_heatmap_with_day_slider(cell_df, value_col, title, colorscale="YlOrRd",
                                          scale=1.0, unit="", ncols=3, day_step=None):
    """Ein Per-Zellen-Gitter pro Spital (Subplots), mit Tages-Slider + Play/Pause."""
    if cell_df is None:
        raise ValueError("Keine Zell-Daten (macro_cell_daily) vorhanden.")
    day_step = day_step or DAY_STEP
    hospitals = sorted(cell_df["hospital_id"].unique())
    n = len(hospitals)
    ncols = min(ncols, n)
    nrows = -(-n // ncols)
    xs = sorted(cell_df["x"].unique())
    ys = sorted(cell_df["y"].unique())
    days = sorted(cell_df["day"].unique())[::day_step]
    zmax = max(1e-9, float(cell_df[value_col].max()) * scale)
    icu_rows = sorted(cell_df.loc[cell_df["department"] == "icu", "y"].unique().tolist())

    def grids(hospital, day):
        d = cell_df[(cell_df["hospital_id"] == hospital) & (cell_df["day"] == day)]
        z = (d.pivot(index="y", columns="x", values=value_col)
               .reindex(index=ys, columns=xs).values) * scale
        npat = (d.pivot(index="y", columns="x", values="total_patients")
                  .reindex(index=ys, columns=xs).values)
        return z, npat[:, :, None]  # 3D customdata -> %{customdata[0]} im Hover

    titles = [h.replace("hospital_", "H") for h in hospitals]
    hov = ("x=%{x}, y=%{y}<br>" + value_col + "=%{z:.2f}" + unit
           + "<br>n=%{customdata[0]:.0f}<extra>%{fullData.name}</extra>")

    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=titles,
                        horizontal_spacing=0.07, vertical_spacing=0.13)
    axpos = []
    for i, h in enumerate(hospitals):
        r, c = i // ncols + 1, i % ncols + 1
        z, npat = grids(h, days[0])
        fig.add_trace(go.Heatmap(z=z, x=xs, y=ys, customdata=npat, coloraxis="coloraxis",
                                 name=titles[i], hovertemplate=hov), row=r, col=c)
        axpos.append((fig.data[-1].xaxis, fig.data[-1].yaxis))

    frames = []
    for day in days:
        data = []
        for i, h in enumerate(hospitals):
            z, npat = grids(h, day)
            data.append(go.Heatmap(z=z, x=xs, y=ys, customdata=npat, coloraxis="coloraxis",
                                   name=titles[i], hovertemplate=hov,
                                   xaxis=axpos[i][0], yaxis=axpos[i][1]))
        frames.append(go.Frame(name=str(day), data=data, traces=list(range(n))))
    fig.frames = frames

    sliders, updatemenus = _slider_and_buttons(days)
    clab = f"{value_col} [{unit}]" if unit else value_col
    fig.update_layout(
        title=f"{title} pro Spital  (ICU-Reihen y={icu_rows}, oben)",
        coloraxis=dict(colorscale=colorscale, cmin=0, cmax=zmax, colorbar=dict(title=clab)),
        width=340 * ncols, height=300 * nrows + 120,
        sliders=sliders, updatemenus=updatemenus,
    )
    fig.update_xaxes(dtick=1)
    fig.update_yaxes(dtick=1, autorange="reversed")  # ICU (y=0) oben, wie department_grid
    return fig

### Per-Spital-Heatmaps (Tages-Slider)

Dieselbe Funktion für verschiedene Grössen.

In [ ]:
hospital_grid_heatmap_with_day_slider(cell_daily, "prevalence", "Carrier-Prävalenz", scale=100, unit="%").show()

In [ ]:
hospital_grid_heatmap_with_day_slider(cell_daily, "carriers", "Carrier-Anzahl", colorscale="Reds").show()

In [ ]:
hospital_grid_heatmap_with_day_slider(cell_daily, "total_patients", "Belegung", colorscale="Blues").show()

### Optional: Einzelnes Spital im Detail

Einzelnes Spital gross, mit demselben Slider.

In [ ]:
def single_hospital_heatmap_with_day_slider(cell_df, value_col, title, hospital,
                                           colorscale="YlOrRd", scale=1.0, unit="", day_step=None):
    day_step = day_step or DAY_STEP
    df = cell_df[cell_df["hospital_id"] == hospital]
    xs = sorted(df["x"].unique())
    ys = sorted(df["y"].unique())
    days = sorted(df["day"].unique())[::day_step]
    zmax = max(1e-9, float(df[value_col].max()) * scale)
    icu_rows = sorted(df.loc[df["department"] == "icu", "y"].unique().tolist())

    def grids(day):
        d = df[df["day"] == day]
        z = (d.pivot(index="y", columns="x", values=value_col)
               .reindex(index=ys, columns=xs).values) * scale
        npat = (d.pivot(index="y", columns="x", values="total_patients")
                  .reindex(index=ys, columns=xs).values)
        return z, npat[:, :, None]

    hov = ("x=%{x}, y=%{y}<br>" + value_col + "=%{z:.2f}" + unit
           + "<br>n=%{customdata[0]:.0f}<extra></extra>")
    clab = f"{value_col} [{unit}]" if unit else value_col
    z0, n0 = grids(days[0])
    frames = [go.Frame(name=str(d), data=[go.Heatmap(
        z=grids(d)[0], x=xs, y=ys, customdata=grids(d)[1], zmin=0, zmax=zmax,
        colorscale=colorscale, hovertemplate=hov, colorbar=dict(title=clab))]) for d in days]
    fig = go.Figure(
        data=[go.Heatmap(z=z0, x=xs, y=ys, customdata=n0, zmin=0, zmax=zmax,
                         colorscale=colorscale, hovertemplate=hov, colorbar=dict(title=clab))],
        frames=frames)
    sliders, updatemenus = _slider_and_buttons(days)
    fig.update_layout(title=f"{title} - {hospital}  (ICU-Reihen y={icu_rows}, oben)",
                      xaxis_title="Gitter-Spalte (x)", yaxis_title="Gitter-Reihe (y)",
                      width=620, height=520, sliders=sliders, updatemenus=updatemenus)
    fig.update_xaxes(dtick=1)
    fig.update_yaxes(dtick=1, autorange="reversed")
    return fig

single_hospital_heatmap_with_day_slider(cell_daily, "prevalence", "Carrier-Prävalenz",
                                        hospital="hospital_001", scale=100, unit="%").show()

## Genotyp-Zusammensetzung (Tages-Slider)

Anderer Plot-Typ, gleiches Slider-Prinzip: Anteil der Resistenzklassen über die Zeit.

In [ ]:
def genotype_bars_with_slider(micro_df, day_step=None):
    day_step = day_step or DAY_STEP
    cols = {"genotype_S_fraction": "S", "genotype_R1_fraction": "R1",
            "genotype_R2_fraction": "R2", "genotype_R3_fraction": "R3",
            "genotype_other_fraction": "other"}
    cols = {c: lbl for c, lbl in cols.items() if c in micro_df.columns}
    names = list(cols.values())
    colors = ["#4c78a8", "#f58518", "#e45756", "#72b7b2", "#bab0ac"][:len(names)]
    days = micro_df["day"].tolist()[::day_step]

    def yvals(day):
        row = micro_df.loc[micro_df["day"] == day].iloc[0]
        return [float(row[c]) for c in cols]

    frames = [go.Frame(name=str(d), data=[go.Bar(x=names, y=yvals(d), marker_color=colors)])
              for d in days]
    fig = go.Figure(data=[go.Bar(x=names, y=yvals(days[0]), marker_color=colors)], frames=frames)
    sliders, updatemenus = _slider_and_buttons(days)
    fig.update_layout(title="Genotyp-Zusammensetzung der Carrier über die Zeit",
                      yaxis=dict(range=[0, 1], title="Anteil"), xaxis_title="Genotyp",
                      width=640, height=480, sliders=sliders, updatemenus=updatemenus)
    return fig

if micro_daily is not None:
    genotype_bars_with_slider(micro_daily).show()

## Zeitreihen (interaktiv, ganzer Verlauf)

Standard-Linienplots über alle Tage — zoom-/hoverbar, kein Slider nötig.

In [ ]:
if macro_daily is not None:
    px.line(macro_daily, x="day", y=["susceptible", "carriers", "isolated_count", "abx_on_count"],
            labels={"value": "Patienten (Anzahl)", "day": "Tag (d)", "variable": "Grösse"},
            title="Populations-Übersicht").show()

In [ ]:
if macro_by_hospital is not None:
    df = macro_by_hospital.copy()
    df["prevalence_pct"] = df["prevalence"] * 100
    px.line(df, x="day", y="prevalence_pct", color="hospital_id",
            labels={"prevalence_pct": "Prävalenz (%)", "day": "Tag (d)", "hospital_id": "Spital"},
            title="Carrier-Prävalenz pro Spital").show()

In [ ]:
if micro_daily is not None:
    m = micro_daily
    fig = go.Figure()
    fig.add_traces([
        go.Scatter(x=m["day"], y=m["p90_resistant_fraction"] * 100, line=dict(width=0),
                   showlegend=False, hoverinfo="skip"),
        go.Scatter(x=m["day"], y=m["p10_resistant_fraction"] * 100, fill="tonexty",
                   line=dict(width=0), fillcolor="rgba(228,87,86,0.2)", name="P10-P90"),
        go.Scatter(x=m["day"], y=m["mean_resistant_fraction"] * 100,
                   line=dict(color="#e45756"), name="Mittel"),
    ])
    fig.update_layout(title="Resistenter Anteil der Carrier über die Zeit",
                      xaxis_title="Tag (d)", yaxis_title="Resistenter Anteil (%)")
    fig.show()

In [ ]:
if micro_daily is not None:
    px.line(micro_daily, x="day", y="mean_n_strains",
            labels={"mean_n_strains": "Stämme (Anzahl)", "day": "Tag (d)"},
            title="Mittlere Stammvielfalt pro Episode").show()